# SAE 204 : BDD - Groupe

Dictionnaire des tables SQL

| table\_name | column\_name | data\_type | is\_nullable |
| :--- | :--- | :--- | :--- |
| controller | controllerid | integer | NO |
| controller | modelid | integer | NO |
| controller | serialnumber | text | NO |
| controlmeasurement | sensorid | integer | NO |
| controlmeasurement | controllerid | integer | NO |
| controlmeasurement | sensortimestamp | timestamp without time zone | NO |
| controlmeasurement | controltimestamp | timestamp without time zone | NO |
| controlmeasurement | controlvalue | double precision | NO |
| model | unit | text | NO |
| model | modelid | integer | NO |
| model | brand | text | NO |
| model | model | text | NO |
| model | name | text | NO |
| sensor | validity | interval | NO |
| sensor | sensorid | integer | NO |
| sensor | serialnumber | text | NO |
| sensor | position | text | NO |
| sensor | modelid | integer | NO |
| sensormeasurement | timestamp | timestamp without time zone | NO |
| sensormeasurement | sensorid | integer | NO |
| sensormeasurement | sensorvalue | double precision | NO |


Tester la connexion avec la base de données

In [ ]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        print("Connection successful!")
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            print(cur.fetchone())
except psycopg.OperationalError as e:
    print(f"Connection failed: {e}")


Script 1 — Erreurs entre mesures manuelles et automatiques :

In [ ]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT sensormeasurement.sensorid,
                       controlmeasurement.controlvalue - sensormeasurement.sensorvalue AS diff
                FROM controlmeasurement
                JOIN sensormeasurement ON controlmeasurement.sensortimestamp = sensormeasurement.timestamp
                WHERE controlmeasurement.sensorid = 1
                ORDER BY controltimestamp
                LIMIT 50;
            """)
            for ligne in cur.fetchall():
                print(f"Capteur {ligne[0]} | Différence : {ligne[1]}")

except psycopg.OperationalError as e:
    print(f"Connexion échouée : {e}")

Script 2 — Moyenne et écart-type :

In [3]:
import psycopg

conn_params = {
    "host": "localhost",
    "dbname": "postgres",
    "user": "postgres",
    "password": "admin"
}

try:
    with psycopg.connect(**conn_params) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT sensorid,
                       AVG(sensorvalue) AS moyenne,
                       STDDEV(sensorvalue) AS ecart_type
                FROM sensormeasurement
                WHERE sensorid = 1
                GROUP BY sensorid;
            """)
            ligne = cur.fetchone()
            print(f"Capteur {ligne[0]} | Moyenne : {ligne[1]:.10f} | Écart-type : {ligne[2]:.10f}")

except psycopg.OperationalError as e:
    print(f"Connexion échouée : {e}")

Capteur 1 | Moyenne : 0.0001409262 | Écart-type : 1.2408729289
